In [1]:
import os
print(os.listdir("/kaggle/input/competitions"))

['global-wheat-detection']


In [2]:
import os
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAME = "wheat"
CLASS_ID = 0

# ----- Data paths -----
# Adjust DATA_DIR to wherever the competition data was downloaded/extracted.
# Expected structure:
#   DATA_DIR/train.csv
#   DATA_DIR/train/*.jpg
#   DATA_DIR/test/*.jpg
DATA_DIR = Path("/kaggle/input/competitions/global-wheat-detection")          # e.g. Path("/kaggle/input/global-wheat-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"

assert TRAIN_CSV.exists(), f"train.csv not found at {TRAIN_CSV.resolve()} - update DATA_DIR"
print("Using data directory:", DATA_DIR.resolve())

Using data directory: /kaggle/input/competitions/global-wheat-detection


In [3]:
all_train_images = sorted(p.stem for p in TRAIN_IMG_DIR.glob("*.jpg"))

def parse_bbox(bbox_str):
    """Parse the string-encoded bbox '[xmin, ymin, w, h]' into floats."""
    return ast.literal_eval(bbox_str)

df_raw = pd.read_csv(TRAIN_CSV)
df = df_raw.copy()
bbox_arr = np.array(df["bbox"].apply(parse_bbox).tolist())
df["x_min"] = bbox_arr[:, 0]
df["y_min"] = bbox_arr[:, 1]
df["box_width"] = bbox_arr[:, 2]
df["box_height"] = bbox_arr[:, 3]

images_with_boxes = set(df["image_id"].unique())
images_without_boxes = sorted(set(all_train_images) - images_with_boxes)

In [4]:
SMALL_AREA_RATIO_THRESH = 0.0005   # box covers < 0.05% of its image
LARGE_AREA_RATIO_THRESH = 0.15     # box covers > 15% of its image
df["area"] = df["box_width"] * df["box_height"]
df["image_area"] = df["width"] * df["height"]
df["area_ratio"] = df["area"] / df["image_area"]
NEG_DIM_MASK = (df["box_width"] <= 0) | (df["box_height"] <= 0)
SMALL_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] < SMALL_AREA_RATIO_THRESH)
LARGE_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] > LARGE_AREA_RATIO_THRESH)
NORMAL_MASK = ~(NEG_DIM_MASK | SMALL_MASK | LARGE_MASK)

In [5]:
clean_df = df.copy()
clean_df["x_max"] = clean_df["x_min"] + clean_df["box_width"]
clean_df["y_max"] = clean_df["y_min"] + clean_df["box_height"]

clean_df["is_negative_dim"] = NEG_DIM_MASK
clean_df["is_small_outlier"] = SMALL_MASK
clean_df["is_large_outlier"] = LARGE_MASK
clean_df["is_outlier"] = NEG_DIM_MASK | SMALL_MASK | LARGE_MASK
clean_df["use_for_training"] = ~(clean_df["is_outlier"])

attribute_cols = [
    "image_id", "width", "height", "source",
    "x_min", "y_min", "box_width", "box_height", "x_max", "y_max", "is_outlier",
    "use_for_training"
]
clean_df = clean_df[attribute_cols]

In [6]:
from sklearn.model_selection import train_test_split

# --- Build an 80/10/10 split, stratified by source ---
image_source_df = clean_df[["image_id", "source"]].drop_duplicates()

train_ids, temp_ids = train_test_split(
    image_source_df["image_id"], test_size=0.20,
    stratify=image_source_df["source"], random_state=RANDOM_SEED,
)
temp_source = image_source_df.set_index("image_id").loc[temp_ids, "source"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_source, random_state=RANDOM_SEED,
)  # 0.5 of the 20% held out -> 10% val, 10% test

# images with no boxes have no known source - split them the same way, unstratified
no_box_ids = sorted(set(all_train_images) - set(image_source_df["image_id"]))
nb_train, nb_temp = train_test_split(no_box_ids, test_size=0.20, random_state=RANDOM_SEED)
nb_val, nb_test = train_test_split(nb_temp, test_size=0.50, random_state=RANDOM_SEED)

split_lookup = {}
for img_id in list(train_ids) + nb_train:
    split_lookup[img_id] = "train"
for img_id in list(val_ids) + nb_val:
    split_lookup[img_id] = "val"
for img_id in list(test_ids) + nb_test:
    split_lookup[img_id] = "test"

print(pd.Series(split_lookup).value_counts())

train    2737
test      343
val       342
Name: count, dtype: int64


In [7]:
YOLO_IMG_DIR = {s: YOLO_DIR / "images" / s for s in ["train", "val", "test"]}
YOLO_LBL_DIR = {s: YOLO_DIR / "labels" / s for s in ["train", "val", "test"]}
for d in list(YOLO_IMG_DIR.values()) + list(YOLO_LBL_DIR.values()):
    d.mkdir(parents=True, exist_ok=True)

def to_yolo_line(row, img_w, img_h):
    x_center = (row.x_min + row.box_width / 2) / img_w
    y_center = (row.y_min + row.box_height / 2) / img_h
    w = row.box_width / img_w
    h = row.box_height / img_h
    x_center, y_center, w, h = (float(np.clip(v, 0, 1)) for v in (x_center, y_center, w, h))
    return f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

boxes_for_yolo = clean_df[clean_df["use_for_training"]]

for img_id in all_train_images:
    split = split_lookup.get(img_id, "train")  # safe now: "train"/"val"/"test" all exist as keys
    rows = boxes_for_yolo[boxes_for_yolo["image_id"] == img_id]

    img_w, img_h = 1024, 1024
    if len(rows) > 0:
        img_w = int(rows.iloc[0]["width"])
        img_h = int(rows.iloc[0]["height"])

    lines = [to_yolo_line(r, img_w, img_h) for r in rows.itertuples()]
    (YOLO_LBL_DIR[split] / f"{img_id}.txt").write_text("\n".join(lines))

    src_img = TRAIN_IMG_DIR / f"{img_id}.jpg"
    dst_img = YOLO_IMG_DIR[split] / f"{img_id}.jpg"
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy(src_img, dst_img)

data_yaml = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: {CLASS_NAME}
"""
(YOLO_DIR / "data.yaml").write_text(data_yaml)

for split in ["train", "val", "test"]:
    n_img = len(list(YOLO_IMG_DIR[split].glob("*.jpg")))
    n_lbl = len(list(YOLO_LBL_DIR[split].glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")

train: 2737 images, 2737 labels
val  : 342 images, 342 labels
test : 343 images, 343 labels


In [8]:
!pip install ultralytics -q
from ultralytics import YOLO

# Load the YOLO26s model
model = YOLO("yolo26s.pt")

# Train the model using default hyperparameters
results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=100,          # Full training run
    imgsz=1024,          # Full resolution for small wheat heads
    batch=4,             # Lowered to prevent Out-Of-Memory crash at 1024px
    project="global_wheat",
    name="train_final",
    exist_ok=True,
    plots=True,          # Generates and saves training curves/plots
    
    # --- Smart Training Settings for 100 Epochs ---
    patience=30,         # Stops training early if mAP doesn't improve for 30 epochs
    cos_lr=True,         # Uses Cosine Learning Rate decay
    close_mosaic=10,     # Disables Mosaic augmentation in the last 10 epochs
    optimizer='AdamW',   # Optimizer choice
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 738.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.158 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/outputs/yolo_dataset/dat

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      3.98G      1.833      1.227   0.007254         23       1024: 100% ━━━━━━━━━━━━ 685/685 5.4it/s 2:070.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 4.9it/s 8.8s0.2s
                   all        342      14917      0.801      0.738      0.818      0.391

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.98G      1.785      1.145   0.007062         36       1024: 100% ━━━━━━━━━━━━ 685/685 5.4it/s 2:070.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 5.3it/s 8.2s0.2s
                   all        342      14917      0.843      0.774      0.841      0.408

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.98G      1.767      1.114   0.006983         71       1024: 100% ━━━━━━━━━━━━ 685/685 5.4it/s 2:070.4ss
                 Class     Images

In [9]:
from ultralytics import YOLO

# Load the best model weights from the final training run
model_path = "/kaggle/working/runs/detect/global_wheat/train_final/weights/best.pt"
model = YOLO(model_path)

# Run validation specifically on the 'val' split
metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", split="val")

# Extract the core metrics from the Ultralytics results object
precision = metrics.box.mp      # Mean Precision
recall = metrics.box.mr         # Mean Recall
mAP_50 = metrics.box.map50      # mAP @ 0.50 IoU
mAP_75 = metrics.box.map75      # mAP @ 0.75 IoU
mAP_50_95 = metrics.box.map     # mAP @ 0.50:0.95 IoU

# Calculate F1 Score (Harmonic mean of Precision and Recall)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Display the results cleanly
print("-" * 30)
print("   Validation Set Metrics   ")
print("-" * 30)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1_score:.4f}")
print(f"mAP@50:     {mAP_50:.4f}")
print(f"mAP@75:     {mAP_75:.4f}")
print(f"mAP@50-95:  {mAP_50_95:.4f}")
print("-" * 30)

Ultralytics 8.4.158 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1826.0±730.6 MB/s, size: 144.0 KB)
val: Scanning /kaggle/working/outputs/yolo_dataset/labels/val.cache... 342 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 342/342 159.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.2it/s 17.7s0.6s
                   all        342      14917      0.923      0.899      0.949      0.566
Speed: 3.9ms preprocess, 34.7ms inference, 0.0ms loss, 5.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
------------------------------
   Validation Set Metrics   
------------------------------
Precision:  0.9228
Recall:     0.8991
F1 Score:   0.9108
mAP@50:     0.9486
mAP@75:     0.5972
mAP@50-95:  0.5661
------------------------------
